In [ ]:
    # Cell 1: Setup & Config (FIXED - dengan seed di awal)
    import os
    import json
    import random
    import numpy as np
    from PIL import Image
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    from torch.utils.data.sampler import BatchSampler
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    from collections import Counter
    from tqdm import tqdm

    # ========== SEED UNTUK REPRODUCIBILITY (HARUS DI AWAL) ==========
    def set_seed(seed=42):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        os.environ['PYTHONHASHSEED'] = str(seed)

    set_seed(42)
    print("Reproducibility seeds set to 42")

    # ========== PATHS ==========
    BASE_PATH = "/content/flood_project/Dataset_Nexus_DolananData"

    def find_mask_dir(base):
        train_dir = os.path.join(base, "train-20260429T073729Z-3-001", "train")
        for name in ["masks", "mask"]:
            if os.path.isdir(os.path.join(train_dir, name)):
                return name
        raise FileNotFoundError("mask/masks folder not found")

    MASK_DIR = find_mask_dir(BASE_PATH)

    TRAIN_IMG  = os.path.join(BASE_PATH, "train-20260429T073729Z-3-001", "train", "img")
    TRAIN_MASK = os.path.join(BASE_PATH, "train-20260429T073729Z-3-001", "train", MASK_DIR)

    VAL_IMG  = os.path.join(BASE_PATH, "validation-20260429T073727Z-3-001", "validation", "img")
    VAL_MASK = os.path.join(BASE_PATH, "validation-20260429T073727Z-3-001", "validation", MASK_DIR)

    TEST_IMG = os.path.join(BASE_PATH, "test-20260429T073643Z-3-001", "test-20260429T073643Z-3-001", "test", "img")

    # ========== CLASS DEFINITIONS ==========
    CLASS_NAMES = {
        0: "background", 1: "building flooded", 2: "building non-flooded",
        3: "grass", 4: "pool", 5: "road flooded",
        6: "road non-flooded", 7: "tree", 8: "vehicle", 9: "water"
    }
    NUM_CLASSES = 10
    EMPTY_CLASSES = {2, 6}  # Tidak pernah muncul di dataset

    # Class weights dari EDA (vehicle dan water paling langka)
    CLASS_WEIGHTS = torch.tensor([0.3, 1.0, 0.0, 0.5, 2.5, 0.8, 0.0, 1.2, 15.0, 8.0], dtype=torch.float32)

    # Normalization dari EDA
    MEAN = [0.4195, 0.4572, 0.3500]
    STD  = [0.2045, 0.1901, 0.2066]

    RARE_CLASSES = {4, 8, 9}  # pool, vehicle, water

    print(f"Train images dir: {TRAIN_IMG}")
    print(f"Val images dir:   {VAL_IMG}")
    print(f"Test images dir:  {TEST_IMG}")
    print(f"Mask folder:      {MASK_DIR}")

Reproducibility seeds set to 42
Train images dir: Dataset_Nexus_DolananData\train-20260429T073729Z-3-001\train\img
Val images dir:   Dataset_Nexus_DolananData\validation-20260429T073727Z-3-001\validation\img
Test images dir:  Dataset_Nexus_DolananData\test-20260429T073643Z-3-001\test-20260429T073643Z-3-001\test\img
Mask folder:      masks


In [3]:
# Cell 2: Lovász-Softmax Loss (FIXED - guard untuk EMPTY_CLASSES)
def lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard

def lovasz_softmax_flat(probas, labels, classes='present'):
    if classes == 'present':
        # FIXED: Filter EMPTY_CLASSES dari perhitungan loss
        all_present = labels.unique()
        classes = [c.item() for c in all_present if c.item() not in EMPTY_CLASSES]
        if len(classes) == 0:
            return torch.tensor(0.0, device=probas.device)
    loss = 0.0
    for c in classes:
        fg = (labels == c).float()
        if fg.sum() == 0:
            continue
        probas_class = probas[:, c]
        errors = (fg - probas_class).abs()
        errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
        fg_sorted = fg[perm]
        loss += torch.dot(errors_sorted, lovasz_grad(fg_sorted))
    # FIXED: divide by max(1, len(classes)) untuk stabilitas
    return loss / max(len(classes), 1)

def lovasz_softmax(probas, labels, ignore=None):
    probas = F.softmax(probas, dim=1)
    B, C, H, W = probas.shape
    probas = probas.permute(0, 2, 3, 1).reshape(-1, C)
    labels = labels.reshape(-1)
    if ignore is not None:
        valid = labels != ignore
        probas = probas[valid]
        labels = labels[valid]
    return lovasz_softmax_flat(probas, labels, classes='present')

class FloodSegLoss(nn.Module):
    def __init__(self, class_weights, lovasz_weight=0.75):
        super().__init__()
        self.lovasz_weight = lovasz_weight
        self.ce_weight = 1.0 - lovasz_weight
        self.ce_loss = nn.CrossEntropyLoss(weight=class_weights, ignore_index=255, reduction='mean')
    
    def forward(self, logits, labels):
        ce = self.ce_loss(logits, labels)
        lovasz = lovasz_softmax(logits, labels, ignore=255)
        total = self.lovasz_weight * lovasz + self.ce_weight * ce
        return total, ce, lovasz

loss_fn = FloodSegLoss(CLASS_WEIGHTS, lovasz_weight=0.75)
print(f"Loss: 0.75*Lovász + 0.25*WCE")
print(f"Weights: {CLASS_WEIGHTS.tolist()}")

Loss: 0.75*Lovász + 0.25*WCE
Weights: [0.30000001192092896, 1.0, 0.0, 0.5, 2.5, 0.800000011920929, 0.0, 1.2000000476837158, 15.0, 8.0]


In [4]:
# Cell 3: Augmentations (sama seperti sebelumnya)
train_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5, border_mode=0),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.GaussNoise(var_limit=(10, 30), p=0.3),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

test_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

# TTA flip
tta_flip_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=1.0),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

print("Augmentations ready.")

Augmentations ready.


c:\Users\Ababil Khoerul Imam\AppData\Local\Programs\Python\Python311\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
C:\Users\Ababil Khoerul Imam\AppData\Local\Temp\ipykernel_3152\1585011157.py:8: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 30), p=0.3),


In [5]:
# Cell 4: Dataset (FIXED - mask validation & cache)
class FloodSegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None, image_ids=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        all_ids = sorted([os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.endswith('.jpg')])
        self.ids = image_ids if image_ids is not None else all_ids
        
        # Cache untuk rare classes (skip tqdm di production)
        self.rare_classes_in_image = {}
        print("Indexing rare classes...")
        for img_id in self.ids:
            mask_path = os.path.join(mask_dir, f"{img_id}.png")
            if os.path.exists(mask_path):
                mask = np.array(Image.open(mask_path))
                # FIXED: Validasi nilai mask
                unique = np.unique(mask)
                invalid = unique[(unique >= NUM_CLASSES) & (unique != 255)]
                if len(invalid) > 0:
                    print(f"WARNING: {img_id} has invalid class values: {invalid}")
                self.rare_classes_in_image[img_id] = set(unique) & RARE_CLASSES
            else:
                self.rare_classes_in_image[img_id] = set()
    
    def __len__(self):
        return len(self.ids)
    
    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img = np.array(Image.open(os.path.join(self.img_dir, f"{img_id}.jpg")).convert('RGB'))
        mask = np.array(Image.open(os.path.join(self.mask_dir, f"{img_id}.png")))
        
        # FIXED: Clamp mask ke range yang valid
        mask = np.clip(mask, 0, NUM_CLASSES - 1)
        
        if self.transform:
            transformed = self.transform(image=img, mask=mask)
            img = transformed['image']
            mask = transformed['mask'].long()
        return img, mask, img_id
    
    def get_rare_indices(self):
        return [i for i, img_id in enumerate(self.ids) if len(self.rare_classes_in_image.get(img_id, set())) > 0]
    
    def get_normal_indices(self):
        rare = set(self.get_rare_indices())
        return [i for i in range(len(self.ids)) if i not in rare]

class FloodTestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.ids = sorted([os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.endswith('.jpg')])
    
    def __len__(self):
        return len(self.ids)
    
    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img = np.array(Image.open(os.path.join(self.img_dir, f"{img_id}.jpg")).convert('RGB'))
        if self.transform:
            img = self.transform(image=img)['image']
        return img, img_id

print("Dataset classes ready.")

Dataset classes ready.


In [6]:
# Cell 5: Balanced Batch Sampler (FIXED - infinite loop)
class BalancedBatchSampler(BatchSampler):
    def __init__(self, dataset, batch_size, rare_ratio=0.35, drop_last=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.rare_ratio = rare_ratio
        self.drop_last = drop_last
        self.rare_indices = dataset.get_rare_indices()
        self.normal_indices = dataset.get_normal_indices()
        print(f"BalancedBatchSampler: {len(self.rare_indices)} rare, {len(self.normal_indices)} normal")
    
    def __iter__(self):
        n_rare = max(1, int(self.batch_size * self.rare_ratio))
        n_normal = self.batch_size - n_rare
        rare_pool = self.rare_indices.copy()
        normal_pool = self.normal_indices.copy()
        random.shuffle(rare_pool)
        random.shuffle(normal_pool)
        i_rare, i_normal = 0, 0
        
        # FIXED: Batch iteration with length
        n_batches = len(self)
        for _ in range(n_batches):
            batch = []
            for _ in range(n_rare):
                if i_rare >= len(rare_pool):
                    random.shuffle(rare_pool)
                    i_rare = 0
                batch.append(rare_pool[i_rare])
                i_rare += 1
            for _ in range(n_normal):
                if i_normal >= len(normal_pool):
                    random.shuffle(normal_pool)
                    i_normal = 0
                batch.append(normal_pool[i_normal])
                i_normal += 1
            random.shuffle(batch)
            yield batch
        
        # Handle remaining if not drop_last
        if not self.drop_last and (i_rare < len(rare_pool) or i_normal < len(normal_pool)):
            batch = []
            while i_rare < len(rare_pool) and len(batch) < self.batch_size:
                batch.append(rare_pool[i_rare])
                i_rare += 1
            while i_normal < len(normal_pool) and len(batch) < self.batch_size:
                batch.append(normal_pool[i_normal])
                i_normal += 1
            if len(batch) > 0:
                yield batch
    
    def __len__(self):
        return len(self.dataset) // self.batch_size

print("Batch sampler ready.")

Batch sampler ready.


In [7]:
# Cell 6: Create Datasets & DataLoaders (FIXED - hanya satu loader, num_workers guard)
train_ids = sorted([os.path.splitext(f)[0] for f in os.listdir(TRAIN_IMG) if f.endswith('.jpg')])
val_ids   = sorted([os.path.splitext(f)[0] for f in os.listdir(VAL_IMG) if f.endswith('.jpg')])

train_dataset = FloodSegDataset(TRAIN_IMG, TRAIN_MASK, transform=train_transform, image_ids=train_ids)
val_dataset   = FloodSegDataset(VAL_IMG, VAL_MASK, transform=val_transform, image_ids=val_ids)
test_dataset  = FloodTestDataset(TEST_IMG, transform=test_transform)

BATCH_SIZE = 8

# FIXED: num_workers guard untuk Windows
num_workers = 4 if os.name != 'nt' else 0

train_sampler = BalancedBatchSampler(train_dataset, batch_size=BATCH_SIZE, rare_ratio=0.35, drop_last=True)

train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers)

print(f"Train: {len(train_dataset)} -> {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} -> {len(val_loader)} batches")
print(f"Test:  {len(test_dataset)} -> {len(test_loader)} batches")

Indexing rare classes...
Indexing rare classes...
BalancedBatchSampler: 927 rare, 517 normal
Train: 1444 -> 180 batches
Val:   448 -> 56 batches
Test:  447 -> 56 batches


In [8]:
# Cell 7: Verify Pipeline
def verify_batch(loader, name, num_batches=2):
    print(f"{name}:")
    for i, (imgs, masks, ids) in enumerate(loader):
        if i >= num_batches:
            break
        print(f"  Batch {i}: imgs {imgs.shape}, masks {masks.shape}, range [{imgs.min():.2f}, {imgs.max():.2f}]")
        unique = torch.unique(masks).tolist()
        has_rare = sum(1 for m in masks if len(set(torch.unique(m).tolist()) & RARE_CLASSES) > 0)
        print(f"    Classes: {unique}, rare in batch: {has_rare}/{len(masks)}")

verify_batch(train_loader, "Train (balanced)")
verify_batch(val_loader, "Val")

Train (balanced):


c:\Users\Ababil Khoerul Imam\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  Batch 0: imgs torch.Size([8, 3, 512, 512]), masks torch.Size([8, 512, 512]), range [-2.41, 3.15]
    Classes: [0, 1, 3, 4, 5, 7], rare in batch: 2/8
  Batch 1: imgs torch.Size([8, 3, 512, 512]), masks torch.Size([8, 512, 512]), range [-2.41, 3.15]
    Classes: [0, 1, 3, 4, 5, 7, 8], rare in batch: 2/8
Val:
  Batch 0: imgs torch.Size([8, 3, 512, 512]), masks torch.Size([8, 512, 512]), range [-2.38, 3.15]
    Classes: [0, 1, 3, 4, 5, 7, 8], rare in batch: 8/8
  Batch 1: imgs torch.Size([8, 3, 512, 512]), masks torch.Size([8, 512, 512]), range [-2.34, 3.15]
    Classes: [0, 1, 3, 4, 5, 7, 8, 9], rare in batch: 6/8


In [9]:
# Cell 8: RLE Utilities
def rle_encode(mask):
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return ' '.join(str(x) for x in runs)

def mask_to_rle(mask, empty_classes=None):
    if empty_classes is None:
        empty_classes = set()
    result = {}
    for cls_id in range(NUM_CLASSES):
        if cls_id in empty_classes:
            result[cls_id] = ''
        else:
            binary = (mask == cls_id).astype(np.uint8)
            result[cls_id] = rle_encode(binary) if binary.sum() > 0 else ''
    return result

print("RLE utilities ready.")

RLE utilities ready.
